# Moontower API Explorer

Scratchpad for hitting every endpoint on `https://api.moontower.ai`.

**Setup** (one-time):
1. `cp .env.example .env` and paste your API key
2. `pip install -r requirements.txt`
3. Run the setup cell below

Each endpoint has its own section — all self-contained, jump to whatever you want.

**API reference:**
- Base: `https://api.moontower.ai/v1`
- Auth: `X-API-Key` header
- Limits: 100 tickers, 31-day range, 30s timeout, 1000 req/min
- Dates: `YYYY-MM-DD`; priority = `trade_date` > `start_date`/`end_date` > latest
- Multi-ticker: repeat the param (`?ticker=SPY&ticker=QQQ`)

## Setup

In [1]:
from datetime import date, timedelta
import pandas as pd
from moontower import call, to_df

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

TICKERS = ["SPY"]
TODAY = date.today().isoformat()
YESTERDAY = (pd.Timestamp.today().normalize() - pd.offsets.BDay(1)).date().isoformat()
_month_ago_ts = pd.Timestamp.today().normalize() - pd.DateOffset(months=1)
if _month_ago_ts.weekday() >= 5:
    _month_ago_ts = _month_ago_ts - pd.offsets.BDay(1)
MONTH_AGO = _month_ago_ts.date().isoformat()

print(f"Ready. TICKERS={TICKERS}  MONTH_AGO={MONTH_AGO}  YESTERDAY={YESTERDAY}  TODAY={TODAY}")

Ready. TICKERS=['SPY']  MONTH_AGO=2026-05-01  YESTERDAY=2026-06-01  TODAY=2026-06-02


## Reference Data

### `/v1/tickers`

Directory of every ticker the API covers. Run this first to see what's available — the result includes symbol, name, asset class, and any grouping tags. Good sanity check that your key works.

In [2]:
tickers_df = to_df(call("tickers"))
print(f"{len(tickers_df)} tickers")
tickers_df.head(20)

431 tickers


,ticker,ticker_id,category,category_id,min_date,max_date
0,AA,403,Metals,9,2007-01-03,None
1,AAL,236,Airlines,29,2013-12-10,None
2,AAPL,92,Technology,13,2007-01-03,None
3,ABBV,223,Pharma,47,2013-01-02,None
4,ABNB,136,Travel,39,None,None
5,ABT,237,Healthcare,23,2007-01-03,None
6,ACHR,423,Transportation,46,2021-09-17,None
7,ADBE,238,Software,32,2007-01-03,None
8,ADI,226,Semiconductors,31,2007-01-03,None
9,AEL,445,Financials,7,2007-01-03,None


## Market Data

### `/v1/price`

OHLCV plus mid. Intraday is 15-min delayed. Accepts a single ticker, multiple tickers (repeat the param), a date range via `start_date`/`end_date`, or a single `trade_date`.

In [3]:
to_df(call("price", ticker=["SPY", "QQQ", "IWM"]))

,ticker,date,snapshot_at,updated_at,open,high,low,close,volume,mid
0,IWM,2026-06-02,2026-06-02T16:01:35,2026-06-02T16:27:11.942000,None,None,None,291.21,None,None
1,QQQ,2026-06-02,2026-06-02T16:03:03,2026-06-02T16:27:51.362000,None,None,None,745.40,None,None
2,SPY,2026-06-02,2026-06-02T16:06:13,2026-06-02T16:29:39.692000,None,None,None,759.64,None,None


In [4]:
to_df(call("price", ticker="SPY", start_date=MONTH_AGO, end_date=TODAY)).tail()

HTTPError: 400 Bad Request on https://api.moontower.ai/v1/price?ticker=SPY&start_date=2026-05-01&end_date=2026-06-02
{"detail":"Date range too large (32 days). Maximum allowed is 31 days. Please use a smaller date range or make multiple requests."}

In [5]:
to_df(call("price", ticker="SPY", trade_date="2025-12-15"))

,ticker,date,snapshot_at,updated_at,open,high,low,close,volume,mid
0,SPY,2025-12-15,2025-12-15T21:00:00,2025-12-16T05:03:38.516000,685.74,685.76,679.25,680.73,90810898.0,682.505


### `/v1/impliedvol`

Full implied-vol surface by delta level for every listed expiry. Narrow to a specific expiry with `expiry_date` — otherwise you'll get every expiry on that trade date.

In [ ]:
iv = to_df(call("impliedvol", ticker="SPY", trade_date="2026-05-19"))
iv.head()

,ticker,date,expiry_date,dte,spot_price,vol100,vol90,vol80,vol70,vol60,vol50,vol40,vol30,vol20,vol10,vol0,snapshot_at,updated_at
0,SPY,2026-05-01,2026-05-01,0,720.59,8.57,8.47,8.38,8.29,8.19,8.10,8.01,7.91,7.82,7.73,7.63,2026-05-01T20:00:04,2026-05-02T01:06:44.294000
1,SPY,2026-05-01,2026-05-04,3,720.59,15.37,11.99,10.70,9.80,9.11,8.56,8.12,7.77,7.51,7.37,7.83,2026-05-01T20:00:04,2026-05-02T01:06:44.294000
2,SPY,2026-05-01,2026-05-05,4,720.59,20.18,14.39,12.62,11.52,10.77,10.20,9.76,9.38,9.04,8.67,7.95,2026-05-01T20:00:04,2026-05-02T01:06:44.294000
3,SPY,2026-05-01,2026-05-06,5,720.59,19.90,14.98,13.32,12.29,11.56,11.00,10.53,10.08,9.61,9.01,7.88,2026-05-01T20:00:04,2026-05-02T01:06:44.294000
4,SPY,2026-05-01,2026-05-07,6,720.59,21.71,15.80,14.00,12.91,12.15,11.57,11.07,10.59,10.06,9.40,8.15,2026-05-01T20:00:04,2026-05-02T01:06:44.294000


In [12]:
to_df(call("impliedvol", ticker="SPY", trade_date="2026-06-01", expiry_date="2026-06-12"))

HTTPError: 504 Gateway Timeout on https://api.moontower.ai/v1/impliedvol?ticker=SPY&trade_date=2026-06-01&expiry_date=2026-06-12
<!DOCTYPE html>
<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->
<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->
<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->
<head>

<title>moontower.ai | 504: Gateway time-out</title>
<meta charset="UTF-8" />
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />
<meta http-equiv="X-UA-Compati

### `/v1/realvol`

Realized volatility at multiple windows (10d/20d/30d/etc).

**⚠️ EOD-only.** Intraday calls return empty — use `trade_date=<yesterday>` or an explicit end-of-day date.

In [8]:
to_df(call("realvol", ticker="SPY", trade_date=YESTERDAY))

,ticker,date,snapshot_at,updated_at,rv_1d,rv_7d,rv_14d,rv_30d,rv_60d,rv_90d,rv_180d,rv_365d
0,SPY,2026-06-01,2026-06-01T20:00:00,2026-06-02T04:30:11.607000,7.021564,5.623224,8.077331,9.316417,10.782009,13.785966,12.331953,9.846204


### `/v1/cmiv`

Constant-maturity implied vol at fixed tenors (30d, 60d, 90d, ...). Use this when you want a clean IV time series that isn't polluted by rolling expiries.

In [13]:
to_df(call("cmiv", ticker="SPY", start_date="2026-05-29", end_date=TODAY)).tail()

,ticker,date,iv_10d,iv_20d,iv_30d,iv_60d,iv_90d,iv_6m,iv_1y,snapshot_at,updated_at
0,SPY,2026-05-29,0.1013,0.1212,0.1255,0.1332,0.1425,0.1562,0.1685,2026-05-29T20:00:05,2026-05-30T01:05:09.401000


### `/v1/ivrank`

IV rank and IV percentile over 1-month, 3-month, and 1-year lookbacks. Fast screener for rich/cheap vol.

In [14]:
to_df(call("ivrank", ticker=["SPY", "QQQ", "IWM", "TLT"]))

,ticker,date,snapshot_at,updated_at,iv,iv_rank_1m,iv_pct_1m,iv_rank_3m,iv_pct_3m,iv_rank_1y,iv_pct_1y
0,TLT,2026-06-02,2026-06-02T15:08:53,2026-06-02T16:24:38.794000,9.15,4.177546,4.761905,2.356406,1.587302,5.870962,3.174603
1,IWM,2026-06-02,2026-06-02T15:07:58,2026-06-02T16:23:59.339000,21.67,44.852941,42.857143,10.932722,26.984127,31.602957,54.761905
2,QQQ,2026-06-02,2026-06-02T14:49:04,2026-06-02T16:25:11.486000,20.94,61.176471,47.619048,23.053892,42.857143,46.293110,75.000000
3,SPY,2026-06-02,2026-06-02T15:09:11,2026-06-02T16:24:41.548000,13.10,21.235521,14.285714,4.122939,4.761905,17.273586,34.920635


### `/v1/rviv`

Realized vol vs implied vol, with the variance risk premium.

VRP = 100 · ln(IV₃₀ / RV₃₀)

Positive VRP = IV above subsequent RV (the usual state for equity indices).

In [16]:
to_df(call("rviv", ticker="SPY", start_date="2026-05-19", end_date=TODAY)).tail()

,ticker,date,snapshot_at,updated_at,rv_30d,atmiv,rv_percentile,vrp
0,SPY,2026-05-19,2026-05-19T20:00:05,2026-05-20T04:54:18.098000,10.358768,15.14,38.955823,37.950694


### `/v1/skew`

10-delta and 25-delta call/put skew plus historical percentiles. Useful for spotting stretched tails.

In [17]:
to_df(call("skew", ticker="SPY", trade_date=MONTH_AGO)).head()

,ticker,date,expiry_date,maturity,snapshot_at,updated_at,skew_10d_call,skew_25d_call,skew_10d_put,skew_25d_put,skew_10d_call_percentile,skew_25d_call_percentile,skew_10d_put_percentile,skew_25d_put_percentile,risk_reversal_10d,risk_reversal_25d,risk_reversal_10d_percentile,risk_reversal_25d_percentile,vol_10d,vol_25d,vol_50d,vol_75d,vol_90d
0,SPY,2026-05-01,2026-05-01,0,2026-05-01T20:00:04,2026-05-02T01:07:05.685000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.077000,0.082000
1,SPY,2026-05-01,2026-05-04,3,2026-05-01T20:00:04,2026-05-02T01:07:05.685000,-0.205438,-0.115815,0.382837,0.186953,54.92,76.63,14.56,22.80,0.588276,0.302769,16.47,21.42,0.068554,0.076286,0.086279,0.102409,0.119310
2,SPY,2026-05-01,2026-05-05,4,2026-05-01T20:00:04,2026-05-02T01:07:05.685000,-0.179200,-0.112620,0.376894,0.165906,72.84,78.94,13.86,16.27,0.556094,0.278526,13.55,16.21,0.084345,0.091187,0.102760,0.119808,0.141489
3,SPY,2026-05-01,2026-05-06,5,2026-05-01T20:00:04,2026-05-02T01:07:05.685000,-0.192998,-0.117377,0.343934,0.152347,63.67,75.27,10.49,11.89,0.536931,0.269724,12.08,14.31,0.088993,0.097332,0.110276,0.127076,0.148203
4,SPY,2026-05-01,2026-05-07,6,2026-05-01T20:00:04,2026-05-02T01:07:05.685000,-0.195081,-0.112656,0.353613,0.156650,62.33,78.92,11.46,13.19,0.548694,0.269306,13.09,14.25,0.093429,0.102996,0.116072,0.134255,0.157117


### `/v1/optionchain`

Full option chain — every strike, every expiry, IV + greeks.

**⚠️ EOD-only** and **heavy**. Always filter by `expiry_date` unless you really want the whole chain. Use `trade_date` for a past snapshot.

In [21]:
chain = to_df(call("optionchain", ticker="SPY", trade_date="2026-05-19", expiry_date="2026-06-12"))
print(chain.shape)
chain.head()

(197, 39)


,trade_date,expiry_date,dte,strike,spot_price,call_bid_price,call_mid_price,call_ask_price,put_bid_price,put_mid_price,put_ask_price,call_volume,put_volume,call_open_interest,put_open_interest,call_bid_size,call_ask_size,put_bid_size,put_ask_size,call_bid_iv,call_mid_iv,call_ask_iv,put_bid_iv,put_mid_iv,put_ask_iv,call_delta,call_gamma,call_theta,call_vega,call_rho,put_delta,put_gamma,put_theta,put_vega,put_rho,snapshot_at,updated_at,ticker,smoothed_vol
0,2026-05-19,2026-06-12,25,400.0,733.8,333.31,334.930,336.55,0.01,0.015,0.02,0.0,10.0,1.0,1234.0,3.0,2.0,970.0,637.0,0.0,1000.0,1000.0,71.34,73.19,75.03,1.0,0.0,0.0,0.0,0.2623,0.0,0.0,0.0,0.0,0.2623,2026-05-19 20:00:05.000000,2026-05-20 01:07:34.232000,SPY,35.92
1,2026-05-19,2026-06-12,25,405.0,733.8,328.33,330.155,331.98,0.01,0.020,0.03,0.0,8.0,0.0,695.0,3.0,1.0,808.0,1029.0,0.0,1000.0,1000.0,69.97,71.78,73.58,1.0,0.0,0.0,0.0,0.2655,0.0,0.0,0.0,0.0,0.2655,2026-05-19 20:00:05.000000,2026-05-20 01:07:34.232000,SPY,35.92
2,2026-05-19,2026-06-12,25,410.0,733.8,323.72,325.130,326.54,0.02,0.025,0.03,0.0,2.0,0.0,703.0,10.0,10.0,145.0,1340.0,0.0,1000.0,1000.0,72.15,73.36,74.57,1.0,0.0,0.0,0.0,0.2688,0.0,0.0,0.0,0.0,0.2688,2026-05-19 20:00:05.000000,2026-05-20 01:07:34.232000,SPY,35.92
3,2026-05-19,2026-06-12,25,415.0,733.8,318.75,320.110,321.47,0.02,0.025,0.03,0.0,0.0,0.0,1072.0,10.0,14.0,303.0,882.0,0.0,1000.0,1000.0,70.74,71.92,73.11,1.0,0.0,0.0,0.0,0.2721,0.0,0.0,0.0,0.0,0.2721,2026-05-19 20:00:05.000000,2026-05-20 01:07:34.232000,SPY,35.92
4,2026-05-19,2026-06-12,25,420.0,733.8,313.38,314.935,316.49,0.02,0.025,0.03,0.0,0.0,0.0,202.0,3.0,10.0,285.0,806.0,0.0,1000.0,1000.0,69.34,70.51,71.67,1.0,0.0,0.0,0.0,0.2754,0.0,0.0,0.0,0.0,0.2754,2026-05-19 20:00:05.000000,2026-05-20 01:07:34.232000,SPY,35.92


### `/v1/cockpit`

Bundle: price, IV, returns, RVIV stats in a single response. Handy for dashboards — avoids N round-trips.

In [20]:
cockpit = to_df(call("cockpit", ticker=["SPY", "QQQ"], start_date=MONTH_AGO, end_date=TODAY))
print(cockpit.shape)
cockpit.tail()

(44, 37)


,ticker,date,close,ma_200d,sd_from_200ma,iv_10d,iv_30d,iv_90d,iv_6m,iv_10d_daily_change,iv_30d_daily_change,iv_30d_weekly_change,iv_30d_monthly_change,term_structure_30d_10d,term_structure_90d_30d,current_price,daily_return,weekly_return,monthly_return,quarterly_return,weekly_annualized_return,monthly_annualized_return,quarterly_annualized_return,weekly_return_zscore,monthly_return_zscore,quarterly_return_zscore,rv_7d,rv_30d,rv_90d,vrp_weekly,vrp_monthly,vrp_quarterly,lagged_vrp_weekly,lagged_vrp_monthly,lagged_vrp_quarterly,snapshot_at,updated_at
39,SPY,2026-05-07,731.58,672.90990,0.516977,0.1245,0.1417,0.1513,0.1617,-0.0038,0.0005,0.0058,-0.0243,0.138153,0.067749,731.58,-0.003066,0.017978,0.082203,0.079632,12.599378,26.200908,13.270959,1.022677,1.578368,0.788985,10.768568,9.890306,14.517221,0.156143,0.432716,0.042210,0.144070,0.678411,0.168203,2026-05-07T20:00:00,2026-05-08T04:01:30.837000
40,SPY,2026-05-06,733.83,672.39630,0.541695,0.1283,0.1412,0.1517,0.1614,0.0007,0.0003,-0.0090,-0.0687,0.100546,0.074363,733.83,0.013899,0.031268,0.113179,0.069427,21.771496,35.560876,11.626016,1.371865,1.694182,0.747596,10.949073,11.886227,14.672490,0.171789,0.187930,0.033908,0.449438,0.765909,0.155512,2026-05-06T20:00:00,2026-05-07T04:02:42.767000
41,SPY,2026-05-05,723.77,671.87100,0.458455,0.1276,0.1409,0.1523,0.1623,-0.0135,-0.0042,-0.0068,-0.0559,0.104232,0.080908,723.77,0.008022,0.016974,0.098402,0.049657,11.901483,31.128631,8.394117,0.801987,1.581739,0.565695,9.550754,11.811063,14.604284,0.336020,0.192949,0.042845,0.553804,0.666234,0.148386,2026-05-05T20:00:00,2026-05-06T04:02:06.851000
42,SPY,2026-05-04,718.01,671.39005,0.414915,0.1411,0.1451,0.1557,0.1618,0.0200,0.0083,-0.0037,-0.0542,0.028349,0.073053,718.01,-0.003663,0.003971,0.094811,0.032499,2.802420,30.042613,5.539434,0.192871,1.507407,0.390637,9.755915,11.752328,14.704212,0.446302,0.234649,0.058880,0.489353,0.695834,0.141805,2026-05-04T20:00:00,2026-05-05T04:02:40.144000
43,SPY,2026-05-01,720.65,670.94020,0.450085,0.1211,0.1368,0.1531,0.1588,-0.0021,0.0009,-0.0152,-0.0656,0.129645,0.119152,720.65,0.002769,0.009399,0.099826,0.041447,6.614742,31.558349,7.034028,0.444539,1.559207,0.481048,8.857088,12.027658,14.706313,0.367267,0.137379,0.041050,0.680010,0.682788,0.146223,2026-05-01T20:00:00,2026-05-02T04:04:15.950000


## Trade Ideas

### `/v1/trade-ideas`

Pre-built trade screens. **Gotcha:** `ideas`, `categories`, and `liquidity_levels` are **comma-separated strings** on this endpoint, not repeated params like `ticker`. Easy to get wrong.

In [22]:
df = to_df(call("trade-ideas"))
print(df.shape)
df.head()

HTTPError: 500 Internal Server Error on https://api.moontower.ai/v1/trade-ideas
Internal Server Error

In [23]:
to_df(call("trade-ideas",
             ticker=["SPY", "QQQ", "IWM", "TLT", "GLD", "USO"],
             liquidity_levels="High,Medium",
             exclude_earnings_weeks=2)).head()

HTTPError: 500 Internal Server Error on https://api.moontower.ai/v1/trade-ideas?ticker=SPY&ticker=QQQ&ticker=IWM&ticker=TLT&ticker=GLD&ticker=USO&liquidity_levels=High%2CMedium&exclude_earnings_weeks=2
Internal Server Error

## Bonus: CSV responses

Most endpoints accept `format=csv`. Use `raw=True` on `call()` to get the raw `Response`, then parse with pandas.

In [26]:
from io import StringIO
resp = call("price", ticker="SPY", start_date="2026-05-19", end_date=TODAY, format="csv", raw=True)
pd.read_csv(StringIO(resp.text)).tail()

,ticker,date,snapshot_at,updated_at,open,high,low,close,volume,mid
0,SPY,2026-05-19,2026-05-19T20:00:00,2026-05-20T04:02:22.483000,734.78,737.65,731.53,733.73,54255900.0,734.59


## Debugging

- **401 / 403** — check `.env` has `MOONTOWER_API_KEY` set and is loaded
- **400** — usually bad date format (`YYYY-MM-DD`) or conflicting date params
- **429** — rate limited; check `X-RateLimit-*` headers below
- **Empty `data`** on intraday `realvol`/`optionchain` — they're EOD-only
- **Unexpected shape on `trade-ideas`** — remember its list params are comma-separated strings

Rate-limit headers on any response:

In [27]:
resp = call("price", ticker="SPY", raw=True)
{k: v for k, v in resp.headers.items() if "ratelimit" in k.lower()}

{}